In [ ]:

import pandas as pd

# 加载训练集和测试集
train_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/train.csv'
test_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/mohs_hardness/test.csv'

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# 检查数据格式和内容
print("训练集前5行:")
print(train_df.head())

print("\n测试集前5行:")
print(test_df.head())

# 检查数据的基本统计信息
print("\n训练集描述:")
print(train_df.describe())

print("\n测试集描述:")
print(test_df.describe())


训练集前5行:
     id  allelectrons_Total  ...  density_Average  Hardness
0  2124                30.0  ...          0.51006       6.0
1   394                64.0  ...          4.74000       3.3
2  3101                97.0  ...          1.79976       5.3
3  1737               151.0  ...          7.77500       1.8
4   561               131.0  ...          1.92652       5.5

[5 rows x 13 columns]

测试集前5行:
     id  allelectrons_Total  ...  density_Average  Hardness
0  3952                80.0  ...          0.86724       5.5
1  4597               239.0  ...          1.86481       7.0
2  9621               152.0  ...          0.51227       2.5
3  8620               232.0  ...          0.59761       6.8
4  4294                46.0  ...          4.01089       2.5

[5 rows x 13 columns]

训练集描述:
                 id  allelectrons_Total  ...  density_Average     Hardness
count   8325.000000         8325.000000  ...      8325.000000  8325.000000
mean    5214.590991          127.784782  ...         2.1400

In [ ]:


# 去除 'id' 列
train_df = train_df.drop(columns=['id'])
test_df = test_df.drop(columns=['id'])

# 拆分特征和目标变量
X_train = train_df.drop(columns=['Hardness'])
y_train = train_df['Hardness']
X_test = test_df.drop(columns=['Hardness'])
y_test = test_df['Hardness']

# 标准化特征
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 将数据转换为 PyTorch 张量
import torch

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# 检查预处理后的数据
print("预处理后的训练集特征前5行:")
print(X_train_tensor[:5])

print("\n预处理后的训练集目标前5个值:")
print(y_train_tensor[:5])

print("\n预处理后的测试集特征前5行:")
print(X_test_tensor[:5])

print("\n预处理后的测试集目标前5个值:")
print(y_test_tensor[:5])



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

t\experiments\OpenInterpreter\venv\lib\site-packages\ipykernel\kernelbase.py", line 545, in dispatch_queue
    await self.process_one()
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\ipykernel\kernelbase.py", line 534, in process_one
    await dispatch(*args)
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\ipykernel\kernelbase.py", line 437, in dispatch_shell
    await result
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\ipykernel\ipkernel.py", line 362, in execute_request
    await super().execute_request(stream, ident, parent)
  File "D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\ipykernel\kernelbase.py", line 778, in execute_request
    reply_content = a

In [ ]:


import torch.nn as nn
import torch.optim as optim

# 定义神经网络模型
class MineralHardnessModel(nn.Module):
    def __init__(self, input_dim):
        super(MineralHardnessModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.tanh = nn.Tanh()

    def forward(self, x):
        x = self.tanh(self.fc1(x))
        x = self.tanh(self.fc2(x))
        x = self.fc3(x)
        return x

# 获取输入特征的维度
input_dim = X_train_tensor.shape[1]

# 实例化模型
model = MineralHardnessModel(input_dim)

# 定义损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

# 检查模型结构
print(model)




MineralHardnessModel(
  (fc1): Linear(in_features=11, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
  (tanh): Tanh()
)


In [ ]:



import torch.nn.functional as F

# 定义训练函数
def train_model(model, criterion, optimizer, X_train, y_train, num_epochs=100, batch_size=32):
    model.train()
    for epoch in range(num_epochs):
        for i in range(0, len(X_train), batch_size):
            X_batch = X_train[i:i+batch_size]
            y_batch = y_train[i:i+batch_size]

            # 前向传播
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)

            # 反向传播和优化
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# 训练模型
train_model(model, criterion, optimizer, X_train_tensor, y_train_tensor, num_epochs=100, batch_size=32)




Epoch [10/100], Loss: 1.3582
Epoch [20/100], Loss: 1.4970
Epoch [30/100], Loss: 1.5223
Epoch [40/100], Loss: 1.5046
Epoch [50/100], Loss: 1.4794
Epoch [60/100], Loss: 1.4383
Epoch [70/100], Loss: 1.3853
Epoch [80/100], Loss: 1.3359
Epoch [90/100], Loss: 1.2882
Epoch [100/100], Loss: 1.2372


In [ ]:



import numpy as np

# 定义评估函数
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        outputs = model(X_test)
        predictions = outputs.numpy()
        true_values = y_test.numpy()
        medae = np.median(np.abs(predictions - true_values))
    return medae

# 计算测试集上的 MedAE
medae = evaluate_model(model, X_test_tensor, y_test_tensor)
print(f'测试集上的 MedAE: {medae:.4f}')


---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
Cell In[6], line 28
     26 pass
     27 print('##active_line14##')
---> 28 medae = evaluate_model(model, X_test_tensor, y_test_tensor)
     29 print('##active_line15##')
     30 print(f'测试集上的 MedAE: {medae:.4f}')

Cell In[6], line 16, in evaluate_model(model, X_test, y_test)
     14 outputs = model(X_test)
     15 print('##active_line8##')
---> 16 predictions = outputs.numpy()
     17 print('##active_line9##')
     18 true_values = y_test.numpy()

RuntimeError: Numpy is not available

In [ ]:



# 定义评估函数
def evaluate_model(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        outputs = model(X_test)
        predictions = outputs.squeeze()
        true_values = y_test.squeeze()
        errors = torch.abs(predictions - true_values)
        medae = torch.median(errors).item()
    return medae

# 计算测试集上的 MedAE
medae = evaluate_model(model, X_test_tensor, y_test_tensor)
print(f'测试集上的 MedAE: {medae:.4f}')



测试集上的 MedAE: 0.7373
